# TopicBank: Bank Creation Experiment

Here we are going to collect interpretable topics (automatically, using topic coherence) from multiple model training.
These topics constitute *topic bank*.
And then the topic bank is going to be used for estimating topic models quality in the notebook [TopicBank-Experiment: Model Validation](TopicBank-Experiment-ModelValidation.ipynb).

The process is repeated for several datasets (some of them are already downloadable using [TopicNet](https://github.com/machine-intelligence-laboratory/TopicNet) library).

# Contents<a id="contents"></a>

* [Data](#data)
    * [Coocs](#coocs)
        * [Lower Memory Consumption (or a Bit of Shamanism. Part 1)](#optimizing-memory)
    * [Documents for Coherence Scores](#docs-for-cohs)
        * [Lower Time Consumption in Case of Big Datasets (or a Bit of Shamanism. Part 2)](#optimizing-time)
* [Experiment](#experiment)
    * [Scores](#scores)
    * [Bank Creation](#bank-creation)
* [Postprocessing](#postprocessing)

In [1]:
# General imports

import dill
import itertools
import json
import numpy as np
import os
import pandas as pd
import sys

from enum import Enum
from scipy.stats import gaussian_kde
from matplotlib import pyplot as plt
from tqdm import tqdm
from typing import (
    Dict,
    Iterable,
)

%matplotlib inline

In [2]:
# Making `topnum` module visible for Python

sys.path.insert(0, '../OptimalNumberOfTopics')

In [3]:
# Optimal number of topics

from topicnet.cooking_machine import Dataset

from topnum.data.vowpal_wabbit_text_collection import VowpalWabbitTextCollection
from topnum.scores import (
    PerplexityScore,
    SparsityPhiScore,
    SparsityThetaScore,
)
from topnum.scores.diversity_score import DiversityScore, KNOWN_METRICS
from topnum.scores._base_coherence_score import (
    SpecificityEstimationMethod,
    TextType,
    WordTopicRelatednessType,
)
from topnum.scores.intratext_coherence_score import ComputationMethod
from topnum.search_methods import TopicBankMethod
from topnum.search_methods.topic_bank.topic_bank import TopicBank
from topnum.search_methods.topic_bank.one_model_train_funcs import (
    default_train_func,

    # Functions below are not used (but could have been)

#     regularization_train_func,
#     specific_initial_phi_train_func,
#     background_topics_train_func,

)

## Data<a id="data"></a>

<div style="text-align: right">Back to <a href=#contents>Contents</a></div>

Loading data from disk, creating batches, dictionary, gathering cooccurrence statistics...

In [4]:
DATA_FOLDER_PATH = '/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/dataset_manager'

In [5]:
sorted(os.listdir(DATA_FOLDER_PATH))

['20NG.csv',
 '20NG__internals',
 'Brown',
 'Brown_BOW.csv',
 'Brown_NOOW.csv',
 'MKB10.csv',
 'MKB10__internals',
 'Reuters',
 'Reuters_BOW.csv',
 'Reuters_NOOW.csv',
 'WikiRef-220',
 '__init__.py',
 '__pycache__',
 'api.py',
 'postnauka.csv',
 'postnauka__internals',
 'ruwiki_good.txt',
 'ruwiki_good__internals',
 'wiki_ref220_bow.csv',
 'wiki_ref220_natural_order.csv']

In [7]:
class DatasetName(Enum):
    POSTNAUKA = 'Post_Science'
    # REUTERS = 'Reuters'
    # BROWN = 'Brown'
    TWENTY_NEWSGROUPS = '20_Newsgroups'
    GOOD_RU_WIKI = 'Good_RU_Wiki'

In [8]:
DATASET_NAME_TO_DATASET_FILE_PATH = {
    DatasetName.POSTNAUKA: os.path.join(
        DATA_FOLDER_PATH, 'postnauka.csv'
    ),
    # DatasetName.REUTERS: os.path.join(
    #     DATA_FOLDER_PATH, 'Reuters.csv'
    # ),
    # DatasetName.BROWN: os.path.join(
    #     DATA_FOLDER_PATH, 'Brown.csv'
    # ),
    DatasetName.TWENTY_NEWSGROUPS: os.path.join(
        DATA_FOLDER_PATH, '20NG.csv'
    ),
    # DatasetName.AG_NEWS: os.path.join(
    #     DATA_FOLDER_PATH, 'AG_News.csv'
    # ),
    # DatasetName.WATAN: os.path.join(
    #     DATA_FOLDER_PATH, 'Watan2004.csv'
    # ),
    # DatasetName.HABRAHABR: os.path.join(
    #     DATA_FOLDER_PATH, 'Habrahabr.csv'
    # ),
    DatasetName.GOOD_RU_WIKI: os.path.join(
        DATA_FOLDER_PATH, 'ruwiki_good.txt'
    ),
}

In [9]:
DATASET_NAME = DatasetName.GOOD_RU_WIKI  # select a dataset here

DATASET_FILE_PATH = DATASET_NAME_TO_DATASET_FILE_PATH[DATASET_NAME]

Checking if all OK with data, what modalities does the collection have.

In [10]:
! head -n 2 $DATASET_FILE_PATH

Санкт-Петербург |@lemmatized год:301 петроград:7 ленинград:18 численность:14 население:33 город:212 россия:32 .:677 федеральный:15 значение:8 административный:5 центр:29 округа:3 ленинградский:16 область:6 основать:5 царь:3 <person>:195 являться:34 столица:19 российский:35 государство:9 назвать:4 честь:6 святой:11 небесный:2 покровитель:2 основатель:3 время:17 стать:25 большой:23 ассоциироваться:1 имя:34 исторически:1 культурно:1 связать:2 рождение:1 империя:7 вхождение:1 современный:7 история:11 роль:5 европейский:3 великий:5 держава:1 расположить:11 страна:16 побережье:4 финский:16 залив:15 устье:4 река:21 нева:31 находиться:17 конституционный:1 суд:5 федерация:14 геральдический:2 совет:8 президент:1 орган:6 власть:11 межпарламентский:3 ассамблея:3 снг:2 разместить:1 главный:9 командование:2 флот:1 штаб:2 западный:8 военный:8 вооружённый:3 сила:6 быть:74 революция:6 февральский:2 октябрьский:6 ход:4 отечественный:4 война:10 блокада:7 результат:12 миллион:37 человек:39 погибнуть:4 объ

In [11]:
def get_dataset_internals_folder_path(dataset_name: DatasetName) -> str:
    return os.path.join('.', dataset_name.value + '__internals')

In [12]:
DATASET_INTERNALS_FOLDER_PATH = get_dataset_internals_folder_path(DATASET_NAME)

In [13]:
DATASET_INTERNALS_FOLDER_PATH

'./Good_RU_Wiki__internals'

In [14]:
%%time

# If using really big datasets (like Habrahabr),
# one may need to set this equal `False`
KEEP_DATASET_IN_MEMORY = True

DATASET = Dataset(
    DATASET_FILE_PATH,
    internals_folder_path=DATASET_INTERNALS_FOLDER_PATH,
    keep_in_memory=KEEP_DATASET_IN_MEMORY,
)

CPU times: user 58.7 s, sys: 2.07 s, total: 1min
Wall time: 31.2 s


Looking what is inside dataset's folder

In [15]:
os.listdir(DATASET_INTERNALS_FOLDER_PATH)

['vw.txt', 'batches', 'dict.dict']

Creating batches

In [16]:
DATASET.get_batch_vectorizer()

artm.BatchVectorizer(data_path="./Good_RU_Wiki__internals/batches", num_batches=9)

In [17]:
os.listdir(DATASET_INTERNALS_FOLDER_PATH)

['vw.txt', 'batches', 'dict.dict']

In [18]:
if KEEP_DATASET_IN_MEMORY:
    DOCUMENTS = list(DATASET._data.index)
else:
    DOCUMENTS = list(DATASET._data_index)

NUM_DOCUMENTS = len(DOCUMENTS)

print(f'Num documents: {NUM_DOCUMENTS}')

Num documents: 8603


Let's look at some text samples

In [19]:
DATASET._data.head()

,vw_text,raw_text,id
id,,,
Санкт-Петербург,Санкт-Петербург |@lemmatized год:301 петроград...,,Санкт-Петербург
Дворцовая_площадь,Дворцовая_площадь |@lemmatized дворцовый:43 пл...,,Дворцовая_площадь
Греко-персидские_войны,Греко-персидские_войны |@lemmatized грёкий:23 ...,,Греко-персидские_войны
Тихий_океан,Тихий_океан |@lemmatized тихий:92 океан:174 ус...,,Тихий_океан
Атлантический_океан,Атлантический_океан |@lemmatized атлантический...,,Атлантический_океан


In [20]:
DATASET.get_possible_modalities()

{'@categories', '@lemmatized', '@ngramms'}

In [21]:
MAIN_MODALITY = '@lemmatized'

In [23]:
DATASET.get_dictionary()

artm.Dictionary(name=5d0a2a95-5ab5-4403-b7e3-01dd229dfc3b, num_entries=892938)

In [25]:
dictionary = DATASET.get_dictionary()

In [27]:
print(dictionary)

for modality in DATASET.get_possible_modalities():
    if modality not in [MAIN_MODALITY]:
        dictionary.filter(class_id=modality, max_df=0, inplace=True)

artm.Dictionary(name=5d0a2a95-5ab5-4403-b7e3-01dd229dfc3b, num_entries=892938)


In [28]:
dictionary.filter(min_df=5, max_df_rate=0.5)

artm.Dictionary(name=5d0a2a95-5ab5-4403-b7e3-01dd229dfc3b, num_entries=61688)

In [29]:
DATASET._cached_dict = dictionary

In [30]:
DATASET.get_dictionary()

artm.Dictionary(name=5d0a2a95-5ab5-4403-b7e3-01dd229dfc3b, num_entries=61688)

In [31]:
import scipy

from typing import List

from topicnet.cooking_machine.models.base_regularizer import BaseRegularizer
from topicnet.cooking_machine.models.thetaless_regularizer import (
    dataset2sparse_matrix,
)
from topicnet.cooking_machine.models import (
    BaseScore as BaseTopicNetScore,
    TopicModel
)

In [32]:
def calc_doc_occurrences(dataset, modality):
    """
    :param n_dw_matrix: sparse document-word matrix, shape is D x W
    :return: sparse matrix of co-occurrences

    doc_occurrences[w1, w2] = the number of the documents
    where there are w1 and w2
    """
    n_dw_matrix = dataset2sparse_matrix(dataset, modality, modalities_to_use=[modality])
    matrix = (scipy.sparse.csc_matrix(n_dw_matrix) > 0).astype(int)
    co_occurrences = matrix.T * matrix

    return co_occurrences.diagonal(), co_occurrences


def create_pmi_top_function(
    doc_occurrences, doc_co_occurrences,
    documents_number, top_sizes,
    topic_indices=None,
    co_occurrences_smooth=1.
):
    """
    :param doc_occurrences: array of doc occurrences of words
    :param doc_co_occurrences: sparse matrix of doc co-occurrences of words
    :param documents_number: number of the documents
    :param top_sizes: list of top values to calculate top-pmi for
    :param co_occurrences_smooth: constant to smooth co-occurrences in log
    :return: function which takes phi and theta and returns
    pair of two arrays: pmi-s of the tops and ppmi-s of the tops

    pmi[i] - pmi(top of size top_sizes[i])
    ppmi[i] - ppmi(top of size top_sizes[i])

    pmi(words) = sum_{u in words, v in words, u != v}
    log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    )

    ppmi(words) = sum_{u in words, v in words, u != v}
    max(log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    ), 0)

    """
    def func(phi):
        T, W = phi.shape
        # T = len(topic_indices)
        topic_indices = list(range(T))

        max_top_size = max(top_sizes)
        topic_pmis, topic_ppmis = dict(), dict()
        pmi, ppmi = np.zeros(max_top_size), np.zeros(max_top_size)
        tops = np.argpartition(phi, -max_top_size, axis=1)[:, -max_top_size:]
        
        for t in topic_indices:
            top = sorted(tops[t], key=lambda w: - phi[t, w])
            # print(top, phi.shape, doc_co_occurrences.shape)
            co_occurrences = doc_co_occurrences[top, :][:, top].todense()
            occurrences = doc_occurrences[top]
            values = np.log(
                (co_occurrences * documents_number + co_occurrences_smooth)
                / (occurrences[:, np.newaxis] * occurrences[np.newaxis, :] + co_occurrences_smooth)
            )
            diag = np.diag_indices(len(values))
            # values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()

            current_pmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_pmis[t] = current_pmi
            pmi += current_pmi

            values[values < 0.] = 0.
            current_ppmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_ppmis[t] = current_ppmi
            ppmi += current_ppmi
            
        sizes = np.arange(2, max_top_size + 1)
        pmi[1:] /= (T * sizes * (sizes - 1))
        ppmi[1:] /= (T * sizes * (sizes - 1))
        indices = np.array(top_sizes) - 1

        for t in topic_indices:
            topic_pmis[t][1:] /= (sizes * (sizes - 1))
            topic_ppmis[t][1:] /= (sizes * (sizes - 1))

        result_topic_pmis = {t: p[indices] for t, p in topic_pmis.items()}
        result_topic_ppmis = {t: p[indices] for t, p in topic_ppmis.items()}

        return pmi[indices], ppmi[indices], result_topic_pmis, result_topic_ppmis

    return func

In [33]:
%%time

occurences, co_occurences = calc_doc_occurrences(DATASET, MAIN_MODALITY)

CPU times: user 51.9 s, sys: 1.66 s, total: 53.6 s
Wall time: 53 s


In [36]:
co_occurences.shape

(61688, 61688)

In [34]:
calc_pmi = create_pmi_top_function(
    occurences, co_occurences,
    DATASET.get_dataset().shape[0], [20],
    topic_indices=[0, 1, 2],
    co_occurrences_smooth=1e-2,
)

In [35]:
import copy


class TopTokenCoherence(BaseTopicNetScore):
    def __init__(self, name, func):
        super().__init__()

        self._name = name
        self.calc_pmi = func

    @property
    def name(self):
        return self._name

    def call(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[1]

    def call_by_topic(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[3]

    def compute(
            self,
            model,
            topics: List[str] = None,
            documents: List[str] = None) -> Dict[str, float]:

        values = self.call_by_topic(model)

        phi = model.get_phi()

        if topics is None:
            topics = list(phi.columns)

            if hasattr(model, 'has_bcg'):
                print(f'Detected bcg topics! Skipping for coherence computation (and will have {len(topics) - 1} topics).')

                topics = topics[:-1]
        else:
            assert False

        index2topic = {phi.columns.get_loc(t): t for t in topics}
        topic2index = {t: i for i, t in index2topic.items()}

        if hasattr(model, 'has_bcg'):
            assert list(index2topic.keys()) == list(values.keys())[:-1]
        else:
            assert list(index2topic.keys()) == list(values.keys())

        result = {
            t: float(values[topic2index[t]])
            for t in topics
        }

        assert len(result) == len(index2topic)

        return result

    def _attach(self, model: TopicModel):
        if self._name in model.custom_scores:
            print(
                f'Score with such name "{self._name}" already attached to model!'
                f' So rewriting it...'
                f' All model\'s custom scores: {list(model.custom_scores.keys())}'
            )

        # TODO: TopicModel should provide ability to add custom scores
        model.custom_scores[self.name] = copy.deepcopy(self)

## Experiment<a id="experiment"></a>

Finally we are getting to the main part!)

### Scores (for Topics and Models)<a id="scores"></a>

<div style="text-align: right">Back to <a href=#contents>Contents</a></div>

Here we define a lot of scores (which mainly differ in initial parameters).

In [37]:
ONE_MODEL_NUM_TOPICS = 20
NUM_TOP_WORDS = 20

In [38]:
top = NUM_TOP_WORDS
target_topic_indices = list(range(ONE_MODEL_NUM_TOPICS))

coherence_score = TopTokenCoherence(
    name=f'coherence_{top}',
    func=create_pmi_top_function(
        occurences, co_occurences,
        DATASET.get_dataset().shape[0], [top],
        # topic_indices=target_topic_indices,
        co_occurrences_smooth=1e-2,
    )
)

diversity_scores = [
    DiversityScore(
        name=f'diversity_{metric}',
        metric=metric,
        class_ids=MAIN_MODALITY,
    )

    for metric in KNOWN_METRICS
]

Other coherence score variations

And a pair of default ARTM scores (these ones are fast)

In [39]:
other_scores = [
    PerplexityScore(
        name='perplexity'
    ),
]

### Bank Creation<a id="bank-creation"></a>

<div style="text-align: right">Back to <a href=#contents>Contents</a></div>

Here we finally run the experiment!

In [40]:
NUM_ITERATIONS = 10

In [41]:
seed = 0

In [42]:
# We use only one train function here
# Other variations are also possible
# It would be even better to make bank using several train functions
# However, it would also take way more time 

TRAIN_FUNCS = default_train_func  # default train func

In [43]:
DATASET_INTERNALS_FOLDER_PATH

'./Good_RU_Wiki__internals'

In [44]:
SEARCH_RESULTS_FOLDER_PATH = os.path.join(
    DATASET_INTERNALS_FOLDER_PATH, 'result'
)

# File with some info about the process
SEARCH_RESULT_FILE_PATH = os.path.join(
    SEARCH_RESULTS_FOLDER_PATH, f'search_result__{seed}.json'
)

# Bank, with topics and their score values
BANK_FOLDER_PATH = os.path.join(
    SEARCH_RESULTS_FOLDER_PATH, f'bank__{seed}'
)

In [45]:
! echo $DATASET_INTERNALS_FOLDER_PATH
! ls -alh $DATASET_INTERNALS_FOLDER_PATH

./Good_RU_Wiki__internals
total 248M
drwxrwxr-x 3 alekseev_v mil_lab 4,0K мар 24 10:57 .
drwxrwxr-x 8 alekseev_v mil_lab 4,0K мар 24 11:03 ..
drwxrwxr-x 2 alekseev_v mil_lab 4,0K мар 24 10:57 batches
-rw-rw-r-- 1 alekseev_v mil_lab  49M мар 24 10:57 dict.dict
-rw-rw-r-- 1 alekseev_v mil_lab 200M мар 24 10:57 vw.txt


In [46]:
SEARCH_RESULTS_FOLDER_PATH

'./Good_RU_Wiki__internals/result'

In [47]:
! ls $SEARCH_RESULTS_FOLDER_PATH

ls: cannot access './Good_RU_Wiki__internals/result': No such file or directory


In [48]:
BANK_FOLDER_PATH

'./Good_RU_Wiki__internals/result/bank__0'

In [49]:
os.makedirs(SEARCH_RESULTS_FOLDER_PATH, exist_ok=True)
os.makedirs(BANK_FOLDER_PATH, exist_ok=True)

In [50]:
seed

0

One cay vary some parameters below (for example `max_num_models` and `num_fit_iterations`).

In [51]:
optimizer = TopicBankMethod(
    data        = DATASET,
    main_modality = MAIN_MODALITY,
    
    min_df_rate = 0.0,  # dictionary filtering has already been done little earlier
    max_df_rate = 1.0,  #   so we don't want these parameters to have any effect

    main_topic_score   = coherence_score,
    other_topic_scores = [],
    other_scores       = [coherence_score] + diversity_scores + other_scores,
   # documents          = TEST_DOCUMENTS,

    start_model_number   = 0,
    max_num_models       = 20,
    one_model_num_topics = ONE_MODEL_NUM_TOPICS,  # 100,
    num_fit_iterations   = NUM_ITERATIONS,  # 100,  # 100 should be enough;
                                 # however, for big data better to reduce this one
                                 # (otherwise the process will be too slow)

    topic_score_threshold_percentile = 90,

    save_bank         = True,
    save_model_topics = True,
    save_file_path    = SEARCH_RESULT_FILE_PATH,
    bank_folder_path  = BANK_FOLDER_PATH,

    train_funcs = TRAIN_FUNCS,
    
    verbose = True,
)

# TODO: use Holdout Perplexity as Stop score

In [52]:
optimizer._result.keys()

dict_keys(['optimum', 'optimum_std', 'bank_scores', 'bank_topic_scores', 'model_scores', 'model_topic_scores', 'num_bank_topics', 'num_model_topics'])

Checking file paths

In [53]:
! echo $DATASET_INTERNALS_FOLDER_PATH
! ls $DATASET_INTERNALS_FOLDER_PATH

./Good_RU_Wiki__internals
batches  dict.dict  result  vw.txt


In [54]:
optimizer._save_file_path

'./Good_RU_Wiki__internals/result/search_result__0.json'

In [55]:
optimizer._topic_bank._path

'./Good_RU_Wiki__internals/result/bank__0'

Fulfilling the search (get ready for a really long process!):

In [56]:
%%time

optimizer.search_for_optimum(DATASET)

100%|█████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.81it/s]


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Bank scores: {'perplexity_score': 37481.07421875, 'coherence_20': 1.2038753315848914, 'diversity_euclidean': 0.05407577211355124, 'diversity_jensenshannon': 0.7089639747186909, 'diversity_hellinger': 0.8184935896289017, 'diversity_cosine': 0.912011199948517, 'perplexity': 37481.07421875, 'ppl_fair': 37481.07421875, 'ppl_cheatty': 8612.4462890625}
100%|█████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.29it/s]


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Bank scores: {'perplexity_score': 37481.07421875, 'coherence_20': 1.2038753315848914, 'diversity_euclidean': 0.05407577211354924, 'diversity_jensenshannon': 0.7089639747167077, 'diversity_hellinger': 0.8184935896267465, 'diversity_cosine': 0.9120111999482552, 'perplexity': 37481.07421875, 'ppl_fair': 37481.07421875, 'ppl_cheatty': 8612.4462890625}
100%|█████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.78it/s]


...

Bank scores: {'perplexity_score': 27877.93359375, 'coherence_20': 1.1562960127004012, 'diversity_euclidean': 0.05066362456618314, 'diversity_jensenshannon': 0.6716496104110566, 'diversity_hellinger': 0.7729775300886383, 'diversity_cosine': 0.8613156239204773, 'perplexity': 27877.93359375, 'ppl_fair': 27877.93359375, 'ppl_cheatty': 7986.11181640625}
100%|█████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.72it/s]


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Bank scores: {'perplexity_score': 27877.93359375, 'coherence_20': 1.1562960127004012, 'diversity_euclidean': 0.050663624566184694, 'diversity_jensenshannon': 0.6716496104122825, 'diversity_hellinger': 0.7729775300901033, 'diversity_cosine': 0.8613156239208036, 'perplexity': 27877.93359375, 'ppl_fair': 27877.93359375, 'ppl_cheatty': 7986.11181640625}
100%|█████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.68it/s]


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Bank scores: {'perplexity_score': 27877.93359375, 'coherence_20': 1.1562960127004012, 'diversity_euclidean': 0.05066362456628404, 'diversity_jensenshannon': 0.6716496104104356, 'diversity_hellinger': 0.7729775301001104, 'diversity_cosine': 0.8613156239197156, 'perplexity': 27877.93359375, 'ppl_fair': 27877.93359375, 'ppl_cheatty': 7986.11181640625}
100%|█████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.51it/s]
Bank scores: {'perplexity_score': 27877.93359375, 'coherence_20': 1.1562960127004012, 'diversity_euclidean': 0.050663624566693856, 'diversity_jensenshannon': 0.6716496104131723, 'diversity_hellinger': 0.7729775300985998, 'diversity_cosine': 0.8613156239216435, 'perplexity': 27877.93359375, 'ppl_fair': 27877.93359375, 'ppl_cheatty': 7986.11181640625}
100%|████████████████████████████████████████████| 20/20 [1:11:30<00:00, 214.53s/it]
CPU times: user 2h 13min 6s, sys: 2min 58s, total: 2h 16min 4s
Wall time: 1h 11min 30s


What topics we have in bank

In [61]:
optimizer._topic_bank.view_topics().head()

topic_0       topic_1   topic_2
@lemmatized рлэ               0.000000e+00  0.000000e+00  0.000013
            локационный       4.034987e-06  0.000000e+00  0.000001
            неподверженность  0.000000e+00  0.000000e+00  0.000000
            даякский          2.344867e-11  1.196141e-14  0.000000
            рэдкот            0.000000e+00  0.000000e+00  0.000000

In [62]:
bank_topics = optimizer._topic_bank.view_topics()

In [63]:
bank_topics.shape

(61688, 3)

In [64]:
bank_topics.head()

topic_0       topic_1   topic_2
@lemmatized рлэ               0.000000e+00  0.000000e+00  0.000013
            локационный       4.034987e-06  0.000000e+00  0.000001
            неподверженность  0.000000e+00  0.000000e+00  0.000000
            даякский          2.344867e-11  1.196141e-14  0.000000
            рэдкот            0.000000e+00  0.000000e+00  0.000000

In [67]:
bank_topics['topic_2'].sort_values(ascending=False)[:20]

@lemmatized  мм            0.012405
             корабль       0.012103
             орудие        0.008174
             крейсер       0.007357
             самолёт       0.006874
             тип           0.005698
             экипаж        0.004602
             проект        0.004583
             борт          0.004550
             установка     0.004326
             флот          0.004275
             башня         0.004271
             скорость      0.004220
             снаряд        0.004213
             вооружение    0.004029
             пушка         0.003621
             ракета        0.003445
             двигатель     0.003368
             система       0.003216
             полёт         0.003076
Name: topic_2, dtype: float64

And topic scores

In [68]:
optimizer._topic_bank.view_topic_scores()

,topic_0,topic_1,topic_2
kernel_size,6147.00000,5204.000000,5652.000000
coherence_20,1.07787,1.329881,1.061137
distance_to_nearest,0.00000,0.915018,0.845497


All models are also saved (topics as $\Phi$ matrices and topic score values)

In [69]:
! ls $optimizer._topic_bank._path

model_0__phi.bin	    model_19__topic_scores.bin
model_0__topic_scores.bin   model_1__phi.bin
model_10__phi.bin	    model_1__topic_scores.bin
model_10__topic_scores.bin  model_2__phi.bin
model_11__phi.bin	    model_2__topic_scores.bin
model_11__topic_scores.bin  model_3__phi.bin
model_12__phi.bin	    model_3__topic_scores.bin
model_12__topic_scores.bin  model_4__phi.bin
model_13__phi.bin	    model_4__topic_scores.bin
model_13__topic_scores.bin  model_5__phi.bin
model_14__phi.bin	    model_5__topic_scores.bin
model_14__topic_scores.bin  model_6__phi.bin
model_15__phi.bin	    model_6__topic_scores.bin
model_15__topic_scores.bin  model_7__phi.bin
model_16__phi.bin	    model_7__topic_scores.bin
model_16__topic_scores.bin  model_8__phi.bin
model_17__phi.bin	    model_8__topic_scores.bin
model_17__topic_scores.bin  model_9__phi.bin
model_18__phi.bin	    model_9__topic_scores.bin
model_18__topic_scores.bin  topics.bin
model_19__phi.bin	    topic_scores.bin


In [70]:
optimizer._result.keys()

dict_keys(['optimum', 'optimum_std', 'bank_scores', 'bank_topic_scores', 'model_scores', 'model_topic_scores', 'num_bank_topics', 'num_model_topics'])

In [71]:
optimizer._result['num_bank_topics']

[2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]

In [72]:
len(optimizer._result['bank_topic_scores'])

20

In [73]:
optimizer._result

{'optimum': 3,
 'optimum_std': 0.0,
 'bank_scores': [{'perplexity_score': 37481.07421875,
   'coherence_20': 1.2038753315848914,
   'diversity_euclidean': 0.05407577211355124,
   'diversity_jensenshannon': 0.7089639747186909,
   'diversity_hellinger': 0.8184935896289017,
   'diversity_cosine': 0.912011199948517,
   'perplexity': 37481.07421875,
   'ppl_fair': 37481.07421875,
   'ppl_cheatty': 8612.4462890625},
  {'perplexity_score': 37481.07421875,
   'coherence_20': 1.2038753315848914,
   'diversity_euclidean': 0.05407577211354924,
   'diversity_jensenshannon': 0.7089639747167077,
   'diversity_hellinger': 0.8184935896267465,
   'diversity_cosine': 0.9120111999482552,
   'perplexity': 37481.07421875,
   'ppl_fair': 37481.07421875,
   'ppl_cheatty': 8612.4462890625},
  {'perplexity_score': 37481.07421875,
   'coherence_20': 1.2038753315848914,
   'diversity_euclidean': 0.054075772113471,
   'diversity_jensenshannon': 0.7089639747137912,
   'diversity_hellinger': 0.8184935896291493,
   

In [74]:
optimizer._result['bank_topic_scores'][0]

[{'kernel_size': 6147,
  'coherence_20': 1.077869629095696,
  'distance_to_nearest': 0.0},
 {'kernel_size': 5204,
  'coherence_20': 1.3298810340740868,
  'distance_to_nearest': 0.9150180765475089}]

In [75]:
optimizer._result['model_topic_scores']

[[{'kernel_size': 6755, 'coherence_20': 0.5358534540606624},
  {'kernel_size': 7235, 'coherence_20': 0.4838309201006973},
  {'kernel_size': 6462, 'coherence_20': 0.4373043160243462},
  {'kernel_size': 5894, 'coherence_20': 0.49186797600324084},
  {'kernel_size': 7138, 'coherence_20': 0.5664375171981993},
  {'kernel_size': 6768, 'coherence_20': 0.4301313148407713},
  {'kernel_size': 6490, 'coherence_20': 0.7604105120520114},
  {'kernel_size': 6742, 'coherence_20': 0.4483818076479467},
  {'kernel_size': 6251, 'coherence_20': 0.4054903770567009},
  {'kernel_size': 6147,
   'coherence_20': 1.077869629095696,
   'distance_to_nearest': 0.0},
  {'kernel_size': 6474, 'coherence_20': 0.6275914617209631},
  {'kernel_size': 5204,
   'coherence_20': 1.3298810340740868,
   'distance_to_nearest': 0.9150180765475089},
  {'kernel_size': 5537, 'coherence_20': 0.8033254592074952},
  {'kernel_size': 5481, 'coherence_20': 0.7138807661406473},
  {'kernel_size': 6024, 'coherence_20': 0.7133950923104643},
  

In [76]:
optimizer._result['model_scores'][0]

{'perplexity_score': 4413.91357421875,
 'coherence_20': 0.5970144894595812,
 'diversity_euclidean': 0.04307072616949526,
 'diversity_jensenshannon': 0.6282176603074374,
 'diversity_hellinger': 0.7115774892444412,
 'diversity_cosine': 0.7898783755509016,
 'perplexity': 4413.91357421875}

In [77]:
optimizer._result['model_scores'][1]

{'perplexity_score': 4408.48046875,
 'coherence_20': 0.6706578405293868,
 'diversity_euclidean': 0.04348795037513734,
 'diversity_jensenshannon': 0.6313844011321076,
 'diversity_hellinger': 0.7159296329104794,
 'diversity_cosine': 0.7950039989385408,
 'perplexity': 4408.48046875}

In [79]:
sum(s['coherence_20'] for s in optimizer._result['bank_topic_scores'][-1]) / 3

1.1562960127004012

In [80]:
len(optimizer._result['bank_scores'])

20

In [81]:
optimizer._result['bank_scores'][-1]

{'perplexity_score': 27877.93359375,
 'coherence_20': 1.1562960127004012,
 'diversity_euclidean': 0.050663624566693856,
 'diversity_jensenshannon': 0.6716496104131723,
 'diversity_hellinger': 0.7729775300985998,
 'diversity_cosine': 0.8613156239216435,
 'perplexity': 27877.93359375,
 'ppl_fair': 27877.93359375,
 'ppl_cheatty': 7986.11181640625}

In [264]:
optimizer._result['bank_topic_scores'][-1]

[{'kernel_size': 6290,
  'coherence_20': 1.9365857260048487,
  'distance_to_nearest': 0.0},
 {'kernel_size': 5987,
  'coherence_20': 2.350993531411413,
  'distance_to_nearest': 0.8670798186186544},
 {'kernel_size': 4904,
  'coherence_20': 1.665684284462788,
  'distance_to_nearest': 0.9068645569155264},
 {'kernel_size': 6419,
  'coherence_20': 3.2077123047347023,
  'distance_to_nearest': 0.9163295596263876}]

In [82]:
import artm
from topnum.model_constructor import KnownModel, init_plsa
from topicnet.cooking_machine.rel_toolbox_lite import (
    count_vocab_size,
    transform_regularizer,
)
from topicnet.cooking_machine.model_constructor import (
    add_standard_scores,
    create_default_topics,
    init_model,
)

def init_model_from_family(
        family: str or KnownModel,
        dataset: Dataset,
        main_modality: str,
        num_topics: int,
        seed: int,
        specific_topic_names = None,
        modalities_to_use: List[str] = None,
        num_processors: int = 3,
        model_params: dict = None,
):
    """
    Returns
    -------
    model: TopicModel() instance
    """
    if isinstance(family, KnownModel):
        family = family.value

    if modalities_to_use is None:
        modalities_to_use = [main_modality]

    custom_regs = {}

    if family == "LDA":
        model = init_lda(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "PLSA":
        model = init_plsa(
            dataset, modalities_to_use, main_modality, num_topics
        )
    elif family == "TARTM":
        model, custom_regs = init_thetaless(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "sparse":
        model = init_bcg_sparse_model(
            dataset, modalities_to_use, main_modality, num_topics, 1, model_params
        )
    elif family == "decorrelation":
        model = init_decorrelated_plsa(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "ARTM":
        model = init_baseline_artm(
            dataset, modalities_to_use, main_modality, num_topics, 1, specific_topic_names, model_params
        )
    else:
        raise ValueError(f'family: {family}')

    model.num_processors = num_processors

    if seed is not None:
        model.seed = seed

    dictionary = dataset.get_dictionary()

    # TODO: maybe this cycle is not necessary
    for modality in dataset.get_possible_modalities():
        if modality not in modalities_to_use:
            dictionary.filter(class_id=modality, max_df=0, inplace=True)

    model.initialize(dictionary)
    add_standard_scores(model, dictionary, main_modality=main_modality,
                        all_modalities=modalities_to_use)

    model = TopicModel(
        artm_model=model,
        custom_regularizers=custom_regs
    )
    model.has_bcg = True  # TODO: only if init_bcg_sparse_model

    return model


def init_bcg_sparse_model(
        dataset,
        modalities_to_use,
        main_modality,
        specific_topics,
        bcg_topics,
        specific_topic_names = None,
        model_params: dict = None
):
    """
    Creates simple artm model with standard scores.

    Parameters
    ----------
    dataset : Dataset
    modalities_to_use : list of str or dict
    main_modality : str
    specific_topics : int
    bcg_topics : int

    Returns
    -------
    model: artm.ARTM() instance
    """
    if model_params is None:
        model_params = dict()

    model = init_plsa(
        dataset, modalities_to_use, main_modality, specific_topics, bcg_topics
    )
    background_topic_names = model.topic_names[-bcg_topics:]

    if specific_topic_names is None:
        print('No spec topics')
        specific_topic_names = model.topic_names[:-bcg_topics]

    dictionary = dataset.get_dictionary()
    baseline_class_ids = {class_id: 1 for class_id in modalities_to_use}
    data_stats = count_vocab_size(dictionary, baseline_class_ids)

    # all coefficients are relative
    regularizers = [
        artm.SmoothSparsePhiRegularizer(
             name='smooth_phi_bcg',
             topic_names=background_topic_names,
             tau=model_params.get("smooth_bcg_tau", 0.1),
             class_ids=[main_modality],
        ),
        artm.SmoothSparseThetaRegularizer(
             name='smooth_theta_bcg',
             topic_names=background_topic_names,
             tau=model_params.get("smooth_bcg_tau", 0.1),
        ),
        artm.SmoothSparsePhiRegularizer(
             name='sparse_phi_sp',
             topic_names=specific_topic_names,
             tau=model_params.get("sparse_sp_tau", -0.05),
             class_ids=[main_modality],
            ),
        artm.SmoothSparseThetaRegularizer(
             name='sparse_theta_sp',
             topic_names=specific_topic_names,
             tau=model_params.get("sparse_sp_tau", -0.05),
        ),
    ]
    for reg in regularizers:
        model.regularizers.add(transform_regularizer(
            data_stats,
            reg,
            model.class_ids,
            n_topics=len(reg.topic_names)
        ))

    return model


def init_baseline_artm(
        dataset,
        modalities_to_use,
        main_modality,
        num_topics,
        bcg_topics,
        specific_topic_names = None,
        model_params: dict = None,
):
    """
    Creates simple artm model with standard scores.

    Parameters
    ----------
    dataset : Dataset
    modalities_to_use : list of str
    main_modality : str
    num_topics : int

    Returns
    -------
    model: artm.ARTM() instance
    """
    if model_params is None:
        model_params = dict()

    model = init_bcg_sparse_model(
        dataset, modalities_to_use, main_modality, num_topics, bcg_topics, specific_topic_names, model_params
    )

    if specific_topic_names is None:
        print('No spec topics')
        specific_topic_names = model.topic_names[:-bcg_topics]

    model.regularizers.add(
        artm.DecorrelatorPhiRegularizer(
            gamma=0,
            tau=model_params.get('decorrelation_tau', 0.01),
            name='decorrelation',
            topic_names=specific_topic_names,
            class_ids=modalities_to_use,
        )
    )

    return model

In [83]:
def artm_train_func(
        dataset: Dataset,
        model_number: int,
        num_topics: int,
        num_fit_iterations: int,
        scores: List = None,
        **kwargs) -> TopicModel:
    """

    Additional Parameters
    ---------------------
    kwargs
        Some params for `_get_topic_model`, such as `cache_theta` and `num_processors`
    """

    topic_model = init_model_from_family(
        family='ARTM',
        dataset=DATASET,
        main_modality=MAIN_MODALITY,
        num_topics=ONE_MODEL_NUM_TOPICS,
        seed=model_number,
        model_params={
            'decorrelation_tau': 0.01,  # best values
            'smooth_bcg_tau': 0.05,
            'sparse_sp_tau': -0.05,
        }
    )

    num_fit_iterations_with_scores = 1

    topic_model._fit(
        dataset.get_batch_vectorizer(),
        num_iterations=max(0, num_fit_iterations - num_fit_iterations_with_scores)
    )
    _fit_model_with_scores(
        topic_model,
        DATASET,
        scores,
        num_fit_iterations=num_fit_iterations_with_scores
    )

    return topic_model


def _fit_model_with_scores(
        topic_model: TopicModel,
        dataset: Dataset,
        scores: List = None,
        num_fit_iterations: int = 1):

    if scores is not None:
        for score in scores:
            score._attach(topic_model)

    topic_model._fit(
        dataset.get_batch_vectorizer(),
        num_iterations=num_fit_iterations
    )

### Bank Creation<a id="bank-creation"></a>

<div style="text-align: right">Back to <a href=#contents>Contents</a></div>

Here we finally run the experiment!

In [84]:
NUM_ITERATIONS = 10

In [85]:
seed = 0

In [86]:
# We use only one train function here
# Other variations are also possible
# It would be even better to make bank using several train functions
# However, it would also take way more time 

TRAIN_FUNCS = artm_train_func  # default train func

In [87]:
DATASET_INTERNALS_FOLDER_PATH

'./Good_RU_Wiki__internals'

In [88]:
SEARCH_RESULTS_FOLDER_PATH = os.path.join(
    DATASET_INTERNALS_FOLDER_PATH, 'result2'
)

# File with some info about the process
SEARCH_RESULT_FILE_PATH = os.path.join(
    SEARCH_RESULTS_FOLDER_PATH, f'search_result__{seed}.json'
)

# Bank, with topics and their score values
BANK_FOLDER_PATH = os.path.join(
    SEARCH_RESULTS_FOLDER_PATH, f'bank__{seed}'
)

In [89]:
SEARCH_RESULTS_FOLDER_PATH

'./Good_RU_Wiki__internals/result2'

In [90]:
BANK_FOLDER_PATH

'./Good_RU_Wiki__internals/result2/bank__0'

In [91]:
os.makedirs(SEARCH_RESULTS_FOLDER_PATH, exist_ok=True)
os.makedirs(BANK_FOLDER_PATH, exist_ok=True)

In [92]:
seed

0

One cay vary some parameters below (for example `max_num_models` and `num_fit_iterations`).

In [93]:
optimizer = TopicBankMethod(
    data        = DATASET,
    main_modality = MAIN_MODALITY,
    
    min_df_rate = 0.0,  # dictionary filtering has already been done little earlier
    max_df_rate = 1.0,  #   so we don't want these parameters to have any effect

    main_topic_score   = coherence_score,
    other_topic_scores = [],
    other_scores       = [coherence_score] + diversity_scores + other_scores,
   # documents          = TEST_DOCUMENTS,

    start_model_number   = 0,
    max_num_models       = 20,
    one_model_num_topics = ONE_MODEL_NUM_TOPICS,  # 100,
    num_fit_iterations   = NUM_ITERATIONS,  # 100,  # 100 should be enough;
                                 # however, for big data better to reduce this one
                                 # (otherwise the process will be too slow)

    topic_score_threshold_percentile = 0.8632394176407866,  # DIFF ALSO HERE

    save_bank         = True,
    save_model_topics = True,
    save_file_path    = SEARCH_RESULT_FILE_PATH,
    bank_folder_path  = BANK_FOLDER_PATH,

    train_funcs = TRAIN_FUNCS,
    
    verbose = True,
)

# TODO: use Holdout Perplexity as Stop score

/home/alekseev_v/projects/iterative/../OptimalNumberOfTopics/topnum/search_methods/topic_bank/topic_bank_method.py:208: UserWarning: topic_score_threshold_percentile 0.8632394176407866 is less than one! It is expected to be in [0, 100]. Are you sure you want to proceed (yes/no)?
  warnings.warn(


In [94]:
optimizer._result.keys()

dict_keys(['optimum', 'optimum_std', 'bank_scores', 'bank_topic_scores', 'model_scores', 'model_topic_scores', 'num_bank_topics', 'num_model_topics'])

Checking file paths

In [95]:
! ls ./Post_Science__internals

batches    _result  _result2  result_unfiltered_dict
dict.dict  result   result2   vw.txt


In [96]:
optimizer._save_file_path

'./Good_RU_Wiki__internals/result2/search_result__0.json'

In [97]:
optimizer._topic_bank._path

'./Good_RU_Wiki__internals/result2/bank__0'

Fulfilling the search (get ready for a really long process!):

In [ ]:
%%time

optimizer.search_for_optimum(DATASET)

  0%|                                                        | 0/20 [00:00<?, ?it/s]No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Detected bcg topics! Skipping for diversity computation (and now 20 topics).
Detected bcg topics! Skipping for diversity computation (and now 20 topics).
Detected bcg topics! Skipping for diversity computation (and now 20 topics).
Detected bcg topics! Skipping for diversity computation (and now 20 topics).

  0%|                                                         | 0/1 [00:00<?, ?it/s]Detected bcg topics! Skipping for coherence computation (and will have 20 topics).

100%|█████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.96it/s]
Skipping saving scores for bcg topic


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Bank scores: {'perplexity_score': 20257.470703125, 'coherence_20': 1.0863093667972832, 'diversity_euclidean': 0.0619528730977368, 'diversity_jensenshannon': 0.7010445276298223, 'diversity_hellinger': 0.8168848068884893, 'diversity_cosine': 0.8948451692109236, 'perplexity': 20257.470703125, 'ppl_fair': 20257.470703125, 'ppl_cheatty': 7362.07666015625}
  5%|██▎                                          | 1/20 [03:30<1:06:32, 210.13s/it]No spec topics
No spec topics


...

Detected bcg topics! Skipping for diversity computation (and now 20 topics).
Detected bcg topics! Skipping for diversity computation (and now 20 topics).
Detected bcg topics! Skipping for diversity computation (and now 20 topics).
Detected bcg topics! Skipping for diversity computation (and now 20 topics).

  0%|                                                         | 0/1 [00:00<?, ?it/s]Detected bcg topics! Skipping for coherence computation (and will have 20 topics).

100%|█████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.73it/s]
Skipping saving scores for bcg topic


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Bank scores: {'perplexity_score': 12956.1708984375, 'coherence_20': 1.0198770555312777, 'diversity_euclidean': 0.0630011660210848, 'diversity_jensenshannon': 0.6894934345925926, 'diversity_hellinger': 0.8011505487472589, 'diversity_cosine': 0.8798738518532896, 'perplexity': 12956.1708984375, 'ppl_fair': 12956.1708984375, 'ppl_cheatty': 6915.63525390625}
 90%|███████████████████████████████████████▌    | 18/20 [1:16:06<08:40, 260.29s/it]No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Detected bcg topics! Skipping for diversity computation (and now 20 topics).
Detected bcg topics! Skipping for diversity computation (and now 20 topics).
Detected bcg topics! Skipping for diversity computation (and now 20 topics).
Detected bcg topics! Skipping for diversity computation (and now 20 topics).

  0%|                                                         | 0/1 [00:00<?, ?it/s]Detected bcg topics! Skipping for coherence computation (and will have 20 topics).

100%|█████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.74it/s]


In [100]:
optimizer._main_modality

'@lemmatized'

What topics we have in bank

In [101]:
optimizer._topic_bank.view_topics().head()

topic_0  topic_1  topic_2  topic_3  topic_4  \
@lemmatized рлэ                   0.0      0.0      0.0      0.0      0.0   
            локационный           0.0      0.0      0.0      0.0      0.0   
            неподверженность      0.0      0.0      0.0      0.0      0.0   
            даякский              0.0      0.0      0.0      0.0      0.0   
            рэдкот                0.0      0.0      0.0      0.0      0.0   

                              topic_5  topic_6  topic_7  topic_8  
@lemmatized рлэ                   0.0      0.0      0.0      0.0  
            локационный           0.0      0.0      0.0      0.0  
            неподверженность      0.0      0.0      0.0      0.0  
            даякский              0.0      0.0      0.0      0.0  
            рэдкот                0.0      0.0      0.0      0.0

In [102]:
bank_topics = optimizer._topic_bank.view_topics()

In [103]:
bank_topics.shape

(61688, 9)

In [104]:
bank_topics.head()

topic_0  topic_1  topic_2  topic_3  topic_4  \
@lemmatized рлэ                   0.0      0.0      0.0      0.0      0.0   
            локационный           0.0      0.0      0.0      0.0      0.0   
            неподверженность      0.0      0.0      0.0      0.0      0.0   
            даякский              0.0      0.0      0.0      0.0      0.0   
            рэдкот                0.0      0.0      0.0      0.0      0.0   

                              topic_5  topic_6  topic_7  topic_8  
@lemmatized рлэ                   0.0      0.0      0.0      0.0  
            локационный           0.0      0.0      0.0      0.0  
            неподверженность      0.0      0.0      0.0      0.0  
            даякский              0.0      0.0      0.0      0.0  
            рэдкот                0.0      0.0      0.0      0.0

In [106]:
bank_topics['topic_5'].sort_values(ascending=False)[:20]

@lemmatized  formula           0.018572
             экспедиция        0.011405
             теория            0.011307
             учёный            0.005120
             научный           0.004637
             исследование      0.004357
             наука             0.004065
             ньютон            0.003989
             система           0.003948
             можно             0.003947
             метод             0.003945
             задача            0.003523
             уравнение         0.003387
             решение           0.003382
             физика            0.003309
             математический    0.003153
             полюс             0.003011
             функция           0.002960
             математика        0.002838
             есть              0.002800
Name: topic_5, dtype: float64

And topic scores

In [107]:
optimizer._topic_bank.view_topic_scores()

,topic_0,topic_1,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7,topic_8
kernel_size,5068.000000,4070.000000,4652.000000,5097.000000,5697.000000,4689.000000,5057.000000,4407.000000,3169.000000
coherence_20,1.029769,1.329881,0.910490,0.873292,0.868489,0.898468,1.015923,1.113694,1.138887
distance_to_nearest,0.000000,0.878302,0.880671,0.792076,0.848455,0.843592,0.861230,0.607785,0.884561


All models are also saved (topics as $\Phi$ matrices and topic score values)

In [108]:
! ls $optimizer._topic_bank._path

model_0__phi.bin	    model_19__topic_scores.bin
model_0__topic_scores.bin   model_1__phi.bin
model_10__phi.bin	    model_1__topic_scores.bin
model_10__topic_scores.bin  model_2__phi.bin
model_11__phi.bin	    model_2__topic_scores.bin
model_11__topic_scores.bin  model_3__phi.bin
model_12__phi.bin	    model_3__topic_scores.bin
model_12__topic_scores.bin  model_4__phi.bin
model_13__phi.bin	    model_4__topic_scores.bin
model_13__topic_scores.bin  model_5__phi.bin
model_14__phi.bin	    model_5__topic_scores.bin
model_14__topic_scores.bin  model_6__phi.bin
model_15__phi.bin	    model_6__topic_scores.bin
model_15__topic_scores.bin  model_7__phi.bin
model_16__phi.bin	    model_7__topic_scores.bin
model_16__topic_scores.bin  model_8__phi.bin
model_17__phi.bin	    model_8__topic_scores.bin
model_17__topic_scores.bin  model_9__phi.bin
model_18__phi.bin	    model_9__topic_scores.bin
model_18__topic_scores.bin  topics.bin
model_19__phi.bin	    topic_scores.bin


In [109]:
optimizer._result.keys()

dict_keys(['optimum', 'optimum_std', 'bank_scores', 'bank_topic_scores', 'model_scores', 'model_topic_scores', 'num_bank_topics', 'num_model_topics'])

In [110]:
optimizer._result['num_bank_topics']

[5, 5, 6, 6, 6, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9]

In [111]:
len(optimizer._result['bank_topic_scores'])

20

In [112]:
optimizer._result

{'optimum': 9,
 'optimum_std': 0.0,
 'bank_scores': [{'perplexity_score': 20257.470703125,
   'coherence_20': 1.0863093667972832,
   'diversity_euclidean': 0.0619528730977368,
   'diversity_jensenshannon': 0.7010445276298223,
   'diversity_hellinger': 0.8168848068884893,
   'diversity_cosine': 0.8948451692109236,
   'perplexity': 20257.470703125,
   'ppl_fair': 20257.470703125,
   'ppl_cheatty': 7362.07666015625},
  {'perplexity_score': 20805.96484375,
   'coherence_20': 1.0628352670334091,
   'diversity_euclidean': 0.06146307847354089,
   'diversity_jensenshannon': 0.7009344857959255,
   'diversity_hellinger': 0.8172129050151774,
   'diversity_cosine': 0.8982419304818527,
   'perplexity': 20805.96484375,
   'ppl_fair': 20805.96484375,
   'ppl_cheatty': 7570.44873046875},
  {'perplexity_score': 18369.23046875,
   'coherence_20': 1.0340908704800558,
   'diversity_euclidean': 0.06003163324628971,
   'diversity_jensenshannon': 0.699532733844726,
   'diversity_hellinger': 0.814769049313561

In [113]:
optimizer._result['bank_topic_scores']

[[{'kernel_size': 5068,
   'coherence_20': 1.0297691355246918,
   'distance_to_nearest': 0.0},
  {'kernel_size': 5007,
   'coherence_20': 1.1756616093491055,
   'distance_to_nearest': 0.9197807889016681},
  {'kernel_size': 4070,
   'coherence_20': 1.3298810340740868,
   'distance_to_nearest': 0.8783019854111281},
  {'kernel_size': 4652,
   'coherence_20': 0.9104903714039343,
   'distance_to_nearest': 0.8806710301929691},
  {'kernel_size': 4803,
   'coherence_20': 0.9857446836345978,
   'distance_to_nearest': 0.8523585978250594}],
 [{'kernel_size': 5068,
   'coherence_20': 1.0297691355246918,
   'distance_to_nearest': 0.0},
  {'kernel_size': 5007,
   'coherence_20': 1.1756616093491055,
   'distance_to_nearest': 0.9197807889016681},
  {'kernel_size': 4070,
   'coherence_20': 1.3298810340740868,
   'distance_to_nearest': 0.8783019854111281},
  {'kernel_size': 4652,
   'coherence_20': 0.9104903714039343,
   'distance_to_nearest': 0.8806710301929691},
  {'kernel_size': 5384,
   'coherence_2

In [114]:
optimizer._result['model_scores'][0]

{'perplexity_score': 4860.8017578125,
 'coherence_20': 0.6750048319629702,
 'diversity_euclidean': 0.05157965876247023,
 'diversity_jensenshannon': 0.6647405644367678,
 'diversity_hellinger': 0.7666404062821004,
 'diversity_cosine': 0.8294436907285049,
 'perplexity': 4860.8017578125}

In [118]:
sum(s['coherence_20'] for s in optimizer._result['bank_topic_scores'][-1]) / 9

1.0198770555312775

In [119]:
len(optimizer._result['bank_scores'])

20

In [120]:
optimizer._result['bank_scores'][-1]

{'perplexity_score': 12956.1708984375,
 'coherence_20': 1.0198770555312777,
 'diversity_euclidean': 0.0630011660210022,
 'diversity_jensenshannon': 0.6894934345906027,
 'diversity_hellinger': 0.8011505487381404,
 'diversity_cosine': 0.8798738518529864,
 'perplexity': 12956.1708984375,
 'ppl_fair': 12956.1708984375,
 'ppl_cheatty': 6915.63525390625}

In [121]:
optimizer._result['model_topic_scores']

[[{'kernel_size': 5842, 'coherence_20': 0.6362738850913598},
  {'kernel_size': 5796, 'coherence_20': 0.7017274275328904},
  {'kernel_size': 4873, 'coherence_20': 0.4808373246649988},
  {'kernel_size': 4946, 'coherence_20': 0.5641984460697425},
  {'kernel_size': 5715, 'coherence_20': 0.5445732395217968},
  {'kernel_size': 5068, 'coherence_20': 0.4520213308964054},
  {'kernel_size': 5068,
   'coherence_20': 1.0297691355246918,
   'distance_to_nearest': 0.0},
  {'kernel_size': 5805, 'coherence_20': 0.5174446740493418},
  {'kernel_size': 5254, 'coherence_20': 0.4274255286876996},
  {'kernel_size': 5007,
   'coherence_20': 1.1756616093491055,
   'distance_to_nearest': 0.9197807889016681},
  {'kernel_size': 5578, 'coherence_20': 0.6489807489743467},
  {'kernel_size': 4070,
   'coherence_20': 1.3298810340740868,
   'distance_to_nearest': 0.8783019854111281},
  {'kernel_size': 4652,
   'coherence_20': 0.9104903714039343,
   'distance_to_nearest': 0.8806710301929691},
  {'kernel_size': 4433, 'c

In [123]:
! echo $SEARCH_RESULTS_FOLDER_PATH
! ls $SEARCH_RESULTS_FOLDER_PATH

./Good_RU_Wiki__internals/result2
bank__0  search_result__0.json


In [124]:
! ls Good_RU_Wiki__internals

batches  dict.dict  result  result2  vw.txt
